# 0 · Dicionário de dados

**CRISP-DM fase 2.2 — descrição dos dados** · Turismo e Hotelaria

---

Este notebook gera um dicionário para a base bruta de reservas de hotel. Para cada
coluna, registra o nome original, uma descrição de negócio, os valores observados
e o tipo semântico.

A base contém reservas de hotéis urbanos e de resort. O campo `is_canceled` indica
o cancelamento; `reservation_status` e `reservation_status_date` descrevem o
estado final registrado para a reserva. Esses campos ficam documentados aqui, mas
são resultados posteriores à reserva e devem ser avaliados com cuidado em qualquer
modelo preditivo para evitar vazamento de informação.

In [8]:
# Importações e caminhos do projeto
from pathlib import Path

import pandas as pd

# O notebook pode ser executado a partir da raiz do projeto ou de notebooks/.
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
ARQUIVO_BRUTO = RAIZ / "data" / "raw" / "hotel_bookings.csv"

if not ARQUIVO_BRUTO.is_file():
    raise FileNotFoundError(f"Base de reservas não encontrada: {ARQUIVO_BRUTO}")

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 100)

print(f"Projeto em {RAIZ}")
print(f"Base bruta: {ARQUIVO_BRUTO.relative_to(RAIZ)}")

Projeto em /home/luan/Área de trabalho/projeto-hotel-clientes/projeto-exemplo-mercado-imobiliario-main
Base bruta: data/raw/hotel_bookings.csv


## Carregando a base de dados

A fonte é `data/raw/hotel_bookings.csv`, mantida sem alterações na camada bruta.
O arquivo reúne reservas de dois tipos de hotel. Valores textuais `NULL`, usados
na origem para representar ausência (por exemplo, em `agent` e `company`), são
convertidos em valores ausentes durante a leitura.

In [9]:
# Carregando a base de reservas. Os campos agent e company são IDs categóricos.
df = pd.read_csv(
    ARQUIVO_BRUTO,
    low_memory=False,
    na_values=["NULL"],
    dtype={"agent": "string", "company": "string"},
)

print(f"{ARQUIVO_BRUTO.relative_to(RAIZ)} — {df.shape[0]:,} linhas × {df.shape[1]} colunas")
df.head()

data/raw/hotel_bookings.csv — 119,390 linhas × 32 colunas


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,meal,country,market_segment,distribution_channel,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,reserved_room_type,assigned_room_type,booking_changes,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,0.0,0,BB,PRT,Direct,Direct,0,0,0,C,C,3,No Deposit,<NA>,<NA>,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,0.0,0,BB,PRT,Direct,Direct,0,0,0,C,C,4,No Deposit,<NA>,<NA>,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,0.0,0,BB,GBR,Direct,Direct,0,0,0,A,C,0,No Deposit,<NA>,<NA>,0,Transient,75.0,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,0.0,0,BB,GBR,Corporate,Corporate,0,0,0,A,A,0,No Deposit,304,<NA>,0,Transient,75.0,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,0.0,0,BB,GBR,Online TA,TA/TO,0,0,0,A,A,0,No Deposit,240,<NA>,0,Transient,98.0,0,1,Check-Out,2015-07-03


## Varredura e classificação das colunas

A classificação combina o tipo armazenado no pandas com regras semânticas para os
campos cujo significado não é apenas numérico:

| campo | tipo atribuído | motivo |
|---|---|---|
| `is_canceled`, `is_repeated_guest` | Binária | indicadores codificados como 0/1 |
| `agent`, `company` | Nominal | códigos identificadores, não medidas numéricas |
| `reservation_status_date` | Data | data do último estado da reserva |
| contagens e quantidades inteiras | Discreta | valores inteiros de eventos ou pessoas |
| `adr` | Contínua | tarifa média diária, com valores decimais |
| demais categorias | Nominal | rótulos sem ordem numérica intrínseca |

Para variáveis categóricas, o dicionário mostra todas as categorias quando há até
20; em campos de alta cardinalidade, informa a quantidade e alguns exemplos.

In [10]:
# Regras semânticas para campos que o dtype, sozinho, classificaria incorretamente.
CAMPOS_BINARIOS = {"is_canceled", "is_repeated_guest"}
CAMPOS_DISCRETOS = {"children"}
CAMPOS_NOMINAIS = {
    "hotel",
    "arrival_date_month",
    "meal",
    "country",
    "market_segment",
    "distribution_channel",
    "agent",
    "company",
    "reserved_room_type",
    "assigned_room_type",
    "deposit_type",
    "customer_type",
    "reservation_status",
}
CAMPO_DATA = "reservation_status_date"


def escalar(valor):
    return valor.item() if hasattr(valor, "item") else valor


def resumir_valores(serie, tipo):
    observados = serie.dropna()
    if observados.empty:
        return []

    if tipo == "Data":
        datas = pd.to_datetime(observados, errors="coerce").dropna()
        if datas.empty:
            return []
        return [datas.min().date().isoformat(), datas.max().date().isoformat()]

    if tipo in {"Nominal", "Binária"}:
        categorias = sorted(observados.astype(str).unique().tolist())
        if len(categorias) <= 20:
            return categorias
        return [f"{len(categorias)} categorias", *categorias[:3]]

    return [escalar(observados.min()), escalar(observados.max())]


col_info = []
for col in df.columns:
    serie = df[col]
    if col == CAMPO_DATA:
        tipo = "Data"
    elif col in CAMPOS_BINARIOS:
        tipo = "Binária"
    elif col in CAMPOS_DISCRETOS:
        tipo = "Discreta"
    elif col in CAMPOS_NOMINAIS:
        tipo = "Nominal"
    elif pd.api.types.is_integer_dtype(serie):
        tipo = "Discreta"
    elif pd.api.types.is_numeric_dtype(serie):
        tipo = "Contínua"
    else:
        tipo = "Nominal"

    col_info.append(
        {
            "variavel": col,
            "descricao": "",
            "valores": resumir_valores(serie, tipo),
            "Tipo": tipo,
        }
    )

In [11]:
# Resumo do dicionário produzido
col_info_df = pd.DataFrame(col_info)

print(f"{len(col_info_df)} colunas descritas")
col_info_df["Tipo"].value_counts()

32 colunas descritas


Tipo
Discreta    15
Nominal     13
Binária      2
Contínua     1
Data         1
Name: count, dtype: int64

## Descrições de negócio

As descrições são associadas pelo nome original da variável, não pela posição da
coluna. Assim, uma mudança na ordem do CSV não desloca as descrições; uma coluna
nova sem definição gera erro explícito para que seja documentada antes da geração
do dicionário.

In [12]:
# Descrições de negócio por nome original da coluna
DESCRICOES = {
    "hotel": "Tipo de hotel da reserva: City Hotel ou Resort Hotel",
    "is_canceled": "Indicador de cancelamento da reserva (1 = cancelada; 0 = não cancelada)",
    "lead_time": "Número de dias entre a data de reserva e a data de chegada",
    "arrival_date_year": "Ano previsto para a chegada",
    "arrival_date_month": "Mês previsto para a chegada",
    "arrival_date_week_number": "Semana do ano prevista para a chegada",
    "arrival_date_day_of_month": "Dia do mês previsto para a chegada",
    "stays_in_weekend_nights": "Número de noites de fim de semana (sábado ou domingo) da estadia",
    "stays_in_week_nights": "Número de noites de semana (segunda a sexta-feira) da estadia",
    "adults": "Número de adultos incluídos na reserva",
    "children": "Número de crianças incluídas na reserva",
    "babies": "Número de bebês incluídos na reserva",
    "meal": "Plano de refeições reservado",
    "country": "País de origem do hóspede, codificado pela norma ISO",
    "market_segment": "Segmento de mercado associado à reserva",
    "distribution_channel": "Canal de distribuição pelo qual a reserva foi realizada",
    "is_repeated_guest": "Indicador de hóspede recorrente (1 = sim; 0 = não)",
    "previous_cancellations": "Número de reservas anteriores canceladas pelo hóspede",
    "previous_bookings_not_canceled": "Número de reservas anteriores não canceladas pelo hóspede",
    "reserved_room_type": "Código do tipo de quarto originalmente reservado",
    "assigned_room_type": "Código do tipo de quarto efetivamente atribuído",
    "booking_changes": "Número de alterações realizadas na reserva até o registro",
    "deposit_type": "Tipo de depósito exigido para a reserva",
    "agent": (
        "Identificador da agência de viagens associada à reserva" " (ausente quando não informado)"
    ),
    "company": "Identificador da empresa associada à reserva (ausente quando não informado)",
    "days_in_waiting_list": "Número de dias em que a reserva permaneceu na lista de espera",
    "customer_type": "Categoria de relacionamento do cliente com o hotel",
    "adr": "Tarifa média diária da reserva (Average Daily Rate)",
    "required_car_parking_spaces": "Número de vagas de estacionamento solicitadas",
    "total_of_special_requests": "Número de solicitações especiais feitas pelo hóspede",
    "reservation_status": (
        "Estado final registrado para a reserva" " (por exemplo, Check-Out, Canceled ou No-Show)"
    ),
    "reservation_status_date": "Data em que o último estado da reserva foi registrado",
}

colunas_sem_descricao = sorted(set(df.columns) - DESCRICOES.keys())
if colunas_sem_descricao:
    raise ValueError(f"Colunas sem descrição de negócio: {colunas_sem_descricao}")

In [13]:
# Associando cada descrição pelo nome da variável
col_info_df["descricao"] = col_info_df["variavel"].map(DESCRICOES)
assert col_info_df["descricao"].notna().all()

col_info_df

,variavel,descricao,valores,Tipo
0,hotel,Tipo de hotel da reserva: City Hotel ou Resort...,"[City Hotel, Resort Hotel]",Nominal
1,is_canceled,Indicador de cancelamento da reserva (1 = canc...,"[0, 1]",Binária
2,lead_time,Número de dias entre a data de reserva e a dat...,"[0, 737]",Discreta
3,arrival_date_year,Ano previsto para a chegada,"[2015, 2017]",Discreta
4,arrival_date_month,Mês previsto para a chegada,"[April, August, December, February, January, J...",Nominal
5,arrival_date_week_number,Semana do ano prevista para a chegada,"[1, 53]",Discreta
6,arrival_date_day_of_month,Dia do mês previsto para a chegada,"[1, 31]",Discreta
7,stays_in_weekend_nights,Número de noites de fim de semana (sábado ou d...,"[0, 19]",Discreta
8,stays_in_week_nights,Número de noites de semana (segunda a sexta-fe...,"[0, 50]",Discreta
9,adults,Número de adultos incluídos na reserva,"[0, 55]",Discreta


## Salvando o dicionário de dados

O resultado é gravado em `data/processed/dicionario.csv`, separado por `;` e sem
índice. A pasta de destino é criada automaticamente se ainda não existir.

In [14]:
# Salvando e conferindo o dicionário de dados
DESTINO = RAIZ / "data" / "processed" / "dicionario.csv"
DESTINO.parent.mkdir(parents=True, exist_ok=True)
col_info_df.to_csv(DESTINO, index=False, sep=";", encoding="utf-8")

print(f"Dicionário gravado em {DESTINO.relative_to(RAIZ)} ({DESTINO.stat().st_size:,} bytes)")
pd.read_csv(DESTINO, sep=";").head(10)

Dicionário gravado em data/processed/dicionario.csv (3,448 bytes)


,variavel,descricao,valores,Tipo
0,hotel,Tipo de hotel da reserva: City Hotel ou Resort...,"['City Hotel', 'Resort Hotel']",Nominal
1,is_canceled,Indicador de cancelamento da reserva (1 = canc...,"['0', '1']",Binária
2,lead_time,Número de dias entre a data de reserva e a dat...,"[0, 737]",Discreta
3,arrival_date_year,Ano previsto para a chegada,"[2015, 2017]",Discreta
4,arrival_date_month,Mês previsto para a chegada,"['April', 'August', 'December', 'February', 'J...",Nominal
5,arrival_date_week_number,Semana do ano prevista para a chegada,"[1, 53]",Discreta
6,arrival_date_day_of_month,Dia do mês previsto para a chegada,"[1, 31]",Discreta
7,stays_in_weekend_nights,Número de noites de fim de semana (sábado ou d...,"[0, 19]",Discreta
8,stays_in_week_nights,Número de noites de semana (segunda a sexta-fe...,"[0, 50]",Discreta
9,adults,Número de adultos incluídos na reserva,"[0, 55]",Discreta
